<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Para modelos baseados em árvores de decisão, como **Random Forest** e **AdaBoost** (com base em árvores), a normalização **não é estritamente necessária**, pois esses modelos são invariantes à escala das características. No entanto, como o pipeline implementado posteriormente realiza a normalização (divisão por 255), isso não prejudica o desempenho e pode ser benéfico caso se deseje comparar com modelos sensíveis à escala (como SVM ou Redes Neurais), onde a normalização é crucial para a convergência do gradiente.

**Solução**:

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    """
    Carrega o dataset Fashion MNIST, converte os rótulos e separa em treino/teste.
    
    Args:
        seed (int): Semente para garantir a reprodutibilidade da divisão.
        
    Returns:
        X_train, X_test, y_train, y_test: Arrays contendo os dados e rótulos.
    """
    print("Carregando Fashion MNIST... Isso pode levar alguns segundos.")
    
    X, y = fetch_openml('Fashion-MNIST', version=1, as_frame=False, return_X_y=True, parser='auto')
    
    y = y.astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    print(f"Dados carregados com sucesso!")
    print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")
    
    return X_train, X_test, y_train, y_test


# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [4]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    """
    Treina um classificador Random Forest.
    """
    rf_model = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    
    print("Treinando Random Forest...")
    rf_model.fit(X_train, y_train)
    
    return rf_model

def train_adaboost(X_train, y_train, seed=42):
    """
    Treina um classificador AdaBoost utilizando Decision Trees como base.
    """
    ada_model = AdaBoostClassifier(n_estimators=50, random_state=seed)
    
    print("Treinando AdaBoost...")
    ada_model.fit(X_train, y_train)
    
    return ada_model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [5]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    """
    Realiza predições em um conjunto de teste e retorna a acurácia.
    
    Args:
        model: O modelo treinado (Random Forest, AdaBoost, etc.).
        X_test: Dados de teste.
        y_test: Rótulos reais de teste.
        
    Returns:
        float: O valor da acurácia (0.0 a 1.0).
    """
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    model_name = model.__class__.__name__
    print(f"Acurácia [{model_name}]: {acc:.4f}")
    
    return acc

A função `evaluate` calcula a acurácia, que representa a proporção de predições corretas em relação ao total. Embora seja uma métrica útil para classes equilibradas como no Fashion MNIST, em alguns casos pode esconder falhas em classes específicas, por isso é recomendado também analisar Precision, Recall e F1-Score.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [6]:
def run_pipeline(model_type="rf", seed=42):
    """
    Orquestra o carregamento, pré-processamento, treinamento e avaliação.
    
    Args:
        model_type (str): "rf" para Random Forest ou "ab" para AdaBoost.
        seed (int): Semente de aleatoriedade para garantir reprodutibilidade.
        
    Returns:
        float: Acurácia final obtida no conjunto de teste.
    """
    X_train, X_test, y_train, y_test = load_data(seed=seed)
    
    X_train_norm = X_train.astype('float32') / 255.0
    X_test_norm = X_test.astype('float32') / 255.0
    
    if model_type == "rf":
        model = train_random_forest(X_train_norm, y_train, seed=seed)
    elif model_type == "ab":
        model = train_adaboost(X_train_norm, y_train, seed=seed)
    else:
        raise ValueError("model_type deve ser 'rf' (Random Forest) ou 'ab' (AdaBoost)")
    
    accuracy = evaluate(model, X_test_norm, y_test)
    
    return accuracy


**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = load_data(seed=42)
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm  = X_test.astype('float32')  / 255.0

depths = [1, 3, 5, 8, 12, 16, 20, None]

print()
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_norm, y_train)
    train_acc = accuracy_score(y_train, dt.predict(X_train_norm))
    test_acc  = accuracy_score(y_test,  dt.predict(X_test_norm))
    diff = train_acc - test_acc
    label = str(d).ljust(4) if d is not None else 'None'
    flag = '  <- arvore completa (memoriza tudo)' if d is None else \
           '  <- overfitting muito alto' if diff > 0.18 else \
           '  <- overfitting alto'       if diff > 0.12 else \
           '  <- overfitting crescendo'  if diff > 0.07 else ''
    print(f'max_depth={label} | Treino: {train_acc:.4f} | Teste: {test_acc:.4f} | Diff: {diff:+.4f}{flag}')


Carregando Fashion MNIST... Isso pode levar alguns segundos.
Dados carregados com sucesso!
Treino: (56000, 784), Teste: (14000, 784)

max_depth=1    | Treino: 0.1993 | Teste: 0.1990 | Diff: +0.0003
max_depth=3    | Treino: 0.5019 | Teste: 0.5016 | Diff: +0.0003
max_depth=5    | Treino: 0.7105 | Teste: 0.7023 | Diff: +0.0082
max_depth=8    | Treino: 0.8033 | Teste: 0.7814 | Diff: +0.0219
max_depth=12   | Treino: 0.8874 | Teste: 0.8143 | Diff: +0.0731  <- overfitting crescendo
max_depth=16   | Treino: 0.9548 | Teste: 0.8106 | Diff: +0.1442  <- overfitting alto
max_depth=20   | Treino: 0.9845 | Teste: 0.8029 | Diff: +0.1817  <- overfitting muito alto
max_depth=None | Treino: 1.0000 | Teste: 0.7946 | Diff: +0.2054  <- arvore completa (memoriza tudo)


O overfitting começa a aparecer a partir de **max_depth ≈ 8**, quando a diferença entre acurácia de treino e teste começa a crescer de forma expressiva. Com max_depth=None, a árvore cresce sem limite e consegue 
100% de acurácia no treino porque cria folhas puras para cada amostra — ela memoriza os dados em vez de generalizar padrões.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?


**Resposta:** o **Random Forest** apresenta um desempenho inicial superior e mais estável no Fashion MNIST em comparação com o AdaBoost básico (sem ajuste fino), devido à sua robustez contra overfitting e à capacidade de capturar padrões complexos através do ensemble de árvores independentes.

**Solução**:

In [11]:
from sklearn.metrics import classification_report, accuracy_score

print("--- Avaliando Random Forest ---")
X_train, X_test, y_train, y_test = load_data(seed=42)
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

rf_model = train_random_forest(X_train_norm, y_train, seed=42)
y_pred_rf = rf_model.predict(X_test_norm)
print("\nRelatório de Classificação Random Forest:")
print(classification_report(y_test, y_pred_rf))

print("\n--- Avaliando AdaBoost ---")
ada_model = train_adaboost(X_train_norm, y_train, seed=42)
y_pred_ada = ada_model.predict(X_test_norm)
print("\nRelatório de Classificação AdaBoost:")
print(classification_report(y_test, y_pred_ada))

--- Avaliando Random Forest ---
Carregando Fashion MNIST... Isso pode levar alguns segundos.
Dados carregados com sucesso!
Treino: (56000, 784), Teste: (14000, 784)
Treinando Random Forest...

Relatório de Classificação Random Forest:
              precision    recall  f1-score   support

           0       0.82      0.87      0.85      1400
           1       0.99      0.96      0.98      1400
           2       0.78      0.81      0.80      1400
           3       0.87      0.92      0.90      1400
           4       0.77      0.84      0.81      1400
           5       0.97      0.97      0.97      1400
           6       0.73      0.57      0.64      1400
           7       0.94      0.94      0.94      1400
           8       0.97      0.97      0.97      1400
           9       0.95      0.96      0.95      1400

    accuracy                           0.88     14000
   macro avg       0.88      0.88      0.88     14000
weighted avg       0.88      0.88      0.88     14000


--- A

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Resposta1:** mudaram ligeiramente. seed 42 = 0.8816 e seed 7 = 0.8794

**Resposta:** Sim, o experimento é reprodutível porque utilizamos o parâmetro `random_state` em todas as funções que envolvem aleatoriedade (como `train_test_split` e a inicialização dos modelos). Ao utilizar a mesma seed, garantimos que a divisão dos dados e o treinamento do modelo ocorram exatamente da mesma forma em diferentes execuções.

**Solução**:

In [12]:
seeds = [42, 7]
results = {}

for s in seeds:
    print(f"\nExecutando pipeline com seed {s}...")
    results[s] = run_pipeline(model_type='rf', seed=s)

for s, acc in results.items():
    print(f"Seed {s}: Acurácia = {acc:.4f}")


Executando pipeline com seed 42...
Carregando Fashion MNIST... Isso pode levar alguns segundos.
Dados carregados com sucesso!
Treino: (56000, 784), Teste: (14000, 784)
Treinando Random Forest...
Acurácia [RandomForestClassifier]: 0.8816

Executando pipeline com seed 7...
Carregando Fashion MNIST... Isso pode levar alguns segundos.
Dados carregados com sucesso!
Treino: (56000, 784), Teste: (14000, 784)
Treinando Random Forest...
Acurácia [RandomForestClassifier]: 0.8794
Seed 42: Acurácia = 0.8816
Seed 7: Acurácia = 0.8794


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

**Resposta:** O RF sofre overfitting. O AdaBoost apresenta underfitting — acurácia de 49% em treino e teste indica que o modelo não generalizou bem.

In [14]:
from sklearn.metrics import accuracy_score
X_train, X_test, y_train, y_test = load_data(seed=42)
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

print("--- Overfitting Check: Random Forest ---")
rf_model = train_random_forest(X_train_norm, y_train, seed=42)
rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train_norm))
rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test_norm))
print(f"RF Acurácia Treino: {rf_train_acc:.4f}")
print(f"RF Acurácia Teste: {rf_test_acc:.4f}")
print(f"Difference (Overfitting): {rf_train_acc - rf_test_acc:.4f}")

print("\n--- Overfitting Check: AdaBoost ---")
ada_model = train_adaboost(X_train_norm, y_train, seed=42)
ada_train_acc = accuracy_score(y_train, ada_model.predict(X_train_norm))
ada_test_acc = accuracy_score(y_test, ada_model.predict(X_test_norm))
print(f"Ada Acurácia Treino: {ada_train_acc:.4f}")
print(f"Ada Acurácia Teste: {ada_test_acc:.4f}")
print(f"Difference (Overfitting): {ada_train_acc - ada_test_acc:.4f}")

Carregando Fashion MNIST... Isso pode levar alguns segundos.
Dados carregados com sucesso!
Treino: (56000, 784), Teste: (14000, 784)
--- Overfitting Check: Random Forest ---
Treinando Random Forest...
RF Acurácia Treino: 1.0000
RF Acurácia Teste: 0.8816
Difference (Overfitting): 0.1184

--- Overfitting Check: AdaBoost ---
Treinando AdaBoost...
Ada Acurácia Treino: 0.4951
Ada Acurácia Teste: 0.4909
Difference (Overfitting): 0.0041


# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?


**Resposta:** O AdaBoost costuma ser mais sensível a mudanças nos hiperparâmetros (como learning_rate e n_estimators) e à qualidade dos dados, pois cada novo estimador tenta corrigir os erros do anterior. Já o Random Forest é mais robusto devido à média de árvores independentes.

In [15]:
n_values = [10, 50, 100]
X_train, X_test, y_train, y_test = load_data(seed=42)
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

print("Variando n_estimators no Random Forest:")
for n in n_values:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train_norm, y_train)
    acc = accuracy_score(y_test, rf.predict(X_test_norm))
    print(f"n_estimators={n}: Acurácia={acc:.4f}")

print("\nVariando n_estimators no AdaBoost:")
for n in n_values:
    ab = AdaBoostClassifier(n_estimators=n, random_state=42)
    ab.fit(X_train_norm, y_train)
    acc = accuracy_score(y_test, ab.predict(X_test_norm))
    print(f"n_estimators={n}: Acurácia={acc:.4f}")

Carregando Fashion MNIST... Isso pode levar alguns segundos.
Dados carregados com sucesso!
Treino: (56000, 784), Teste: (14000, 784)
Variando n_estimators no Random Forest:
n_estimators=10: Acurácia=0.8623
n_estimators=50: Acurácia=0.8796
n_estimators=100: Acurácia=0.8816

Variando n_estimators no AdaBoost:
n_estimators=10: Acurácia=0.3131
n_estimators=50: Acurácia=0.4909
n_estimators=100: Acurácia=0.5729


# Questão 9

1. **A acurácia é suficiente?** 
Não. Em problemas multiclasse como o Fashion MNIST, a acurácia pode esconder o fato de o modelo confundir classes visualmente similares (como 't-shirt' e 'shirt'). Métricas como Precisão, Recall e F1-Score (e a matriz de confusão) são essenciais para entender onde o modelo falha.

2. **Como garantir que o resultado não foi acaso?** 
Utilizando técnicas como **Validação Cruzada (Cross-Validation)**, que divide os dados em múltiplos conjuntos de treino/teste, e repetindo os experimentos com diferentes sementes aleatórias para verificar a variância dos resultados.

3. **Problemas metodológicos:** 
A falta de um conjunto de **validação** para ajuste de hiperparâmetros (estamos testando direto no teste) e a ausência de análise estatística (como desvio padrão das acurácias) para confirmar se a diferença entre RF e AB é significativa.

4. **O pipeline é confiável?** 
Ele é um bom ponto de partida e garante **reprodutibilidade**, mas para ser considerado 'confiável' em produção, precisaria de cross-validation, busca exaustiva de hiperparâmetros (GridSearch) e monitoramento de métricas além da acurácia global.